In [3]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [4]:
DATASET_NAME = "Whisper Base 0.5s"

DATASET_PATH = "/Users/bhavaykhatri/Desktop/whisper_base/singBAP_dataset_whisper_whisper-base_0.5s.parquet"

df = pd.read_parquet(DATASET_PATH)

print(df.shape)
df.head()

(34409, 13)


,condition,experience,extractor,filename,filepath,frame_index,phonation,scale,singer,take,embedding,embedding_dtype,embedding_shape
0,after_instruction,inexperienced,whisper_whisper-base,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,0,,glissando,INEX-1,1,b'\xa6\xdf\x19\xbe7YJ\xbf\xc3^\x8a>{U\xa1\xbf\...,<f4,[512]
1,after_instruction,inexperienced,whisper_whisper-base,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,1,,glissando,INEX-1,1,b'\x98A\x0b\xbcGj@\xbf\xd6~\xcc>\xf4\xe6v\xbfA...,<f4,[512]
2,after_instruction,inexperienced,whisper_whisper-base,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,2,,glissando,INEX-1,1,b'\xcb\x1f\x0e\xbek\x87e\xbf\x13\xd0\xbc>V@\x9...,<f4,[512]
3,after_instruction,inexperienced,whisper_whisper-base,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,3,,glissando,INEX-1,1,b'\xfbx0\xbeW;R\xbfV\xe8\x9a>\x14\xd4\xa0\xbf\...,<f4,[512]
4,after_instruction,inexperienced,whisper_whisper-base,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,4,,glissando,INEX-1,1,b'F\x0bI\xbea\x8fs\xbf\xa0\x8c\x9e>8R\x99\xbfk...,<f4,[512]


In [5]:
TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

df = df[
    df["experience"].isin(
        ["intermediate", "professional"]
    )
].copy()

df = df[
    df["condition"].isin(TARGET_CLASSES)
].copy()

print(df.shape)
print(df["condition"].value_counts())

(28418, 13)
condition
correct               5137
hunched_back          4540
sideways              4285
chest_breathing       4259
over_articulation     3454
under_articulation    3381
arched_back           3362
Name: count, dtype: int64


In [6]:
def decode_embedding(x):

    if isinstance(
        x,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            x,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        x,
        dtype=np.float32,
    ).reshape(-1)

In [7]:
X = np.vstack(
    df["embedding"].apply(
        decode_embedding
    )
)

y = df["condition"].astype(str).to_numpy()

groups = df["filename"].astype(str).to_numpy()

print(X.shape)
print(y.shape)
print("Unique recordings:", len(np.unique(groups)))

(28418, 512)
(28418,)
Unique recordings: 3046


In [8]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

print(X_train.shape)
print(X_test.shape)

print(
    "Shared recordings:",
    len(
        set(groups[train_idx])
        &
        set(groups[test_idx])
    )
)

(22735, 512)
(5683, 512)
Shared recordings: 0


In [9]:
MODELS = {

    "MLP":

    make_pipeline(

        StandardScaler(),

        MLPClassifier(

            hidden_layer_sizes=(256,128),

            early_stopping=True,

            max_iter=300,

            random_state=42,

        )

    ),

    "KNN":

    make_pipeline(

        StandardScaler(),

        KNeighborsClassifier(

            n_neighbors=15,

            metric="cosine",

            n_jobs=-1,

        )

    ),

    "Random Forest":

    RandomForestClassifier(

        n_estimators=300,

        class_weight="balanced",

        random_state=42,

        n_jobs=-1,

    ),

    "Linear SVM":

    make_pipeline(

        StandardScaler(),

        LinearSVC(

            class_weight="balanced",

            max_iter=10000,

            random_state=42,

        )

    ),
}

In [10]:
results = []

for model_name, base_model in MODELS.items():

    print(f"Training {model_name}...")

    model = clone(base_model)

    start = time.time()

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    elapsed = time.time() - start

    results.append({

        "Embedding": DATASET_NAME,

        "Model": model_name,

        "Accuracy":

            accuracy_score(
                y_test,
                predictions,
            ),

        "Balanced Accuracy":

            balanced_accuracy_score(
                y_test,
                predictions,
            ),

        "Macro F1":

            f1_score(

                y_test,

                predictions,

                average="macro",

            ),

        "Train/Eval Time":

            elapsed,

    })

Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [11]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False,
)

results_df

,Embedding,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time
0,Whisper Base 0.5s,MLP,0.375330,0.379603,0.375847,18.745547
2,Whisper Base 0.5s,Random Forest,0.298610,0.311990,0.297093,16.750727
3,Whisper Base 0.5s,Linear SVM,0.298786,0.316737,0.291247,69.733542
1,Whisper Base 0.5s,KNN,0.268168,0.268770,0.273406,1.731474


In [12]:
results_df.to_csv(
    "whisper_base_0.5s_baseline.csv",
    index=False,
)

print("Saved.")

Saved.
